# LeetCode 2094 — Finding 3-Digit Even Numbers bằng Python

Link đề: https://leetcode.com/problems/finding-3-digit-even-numbers/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Sorting, Enumeration

```python
class Solution:
    def findEvenNumbers(self, digits: List[int]) -> List[int]:

```

## 1. Mục tiêu học

Sau bài này bạn sẽ:

1. Hiểu cách **sinh số từ các chữ số** cho trước: không dùng lại quá số lần có, không số 0 đứng đầu, chữ số cuối chẵn.
2. Biết dùng `itertools.permutations`, `set`, `sorted` và `collections.Counter`.
3. Nắm kỹ thuật **liệt kê không gian đáp án** (chỉ 450 số chẵn có 3 chữ số) thay vì liệt kê hoán vị của đầu vào.
4. So sánh 4 cách: ba vòng lặp lồng nhau, `permutations`, đếm tần suất, backtracking.
5. Biết trace, viết test đối chiếu ngẫu nhiên và tránh các lỗi quen thuộc (số trùng, số 0 đầu).

## 2. Đề bài & ví dụ

Cho mảng `digits` gồm các chữ số (0–9, có thể trùng). Hãy chọn **ba phần tử ở ba vị trí khác nhau** rồi ghép chúng theo **thứ tự bất kỳ** thành một số có 3 chữ số. Số tạo ra phải thỏa:

- **không có chữ số 0 ở đầu** (nên nằm trong `[100, 999]`),
- là **số chẵn**.

Trả về mảng **tăng dần, không trùng** gồm mọi số như vậy.

| Ví dụ | digits | Output |
|---|---|---|
| 1 | `[2,1,3,0]` | `[102,120,130,132,210,230,302,310,312,320]` |
| 2 | `[2,2,8,8,2]` | `[222,228,282,288,822,828,882]` |
| 3 | `[3,7,5]` | `[]` |

Constraints: `3 <= digits.length <= 100`, `0 <= digits[i] <= 9`.

Lưu ý ví dụ 2: chỉ có ba chữ số `2` và hai chữ số `8` nên `888` **không** tạo được (cần ba chữ số `8`).

In [1]:
digits = [2, 1, 3, 0]
expected = [102, 120, 130, 132, 210, 230, 302, 310, 312, 320]
print("digits =", digits)
print("Số lượng đáp án kỳ vọng:", len(expected))
print("Tất cả đều chẵn?", all(x % 2 == 0 for x in expected))
print("Tất cả có 3 chữ số?", all(100 <= x <= 999 for x in expected))

digits = [2, 1, 3, 0]
Số lượng đáp án kỳ vọng: 10
Tất cả đều chẵn? True
Tất cả có 3 chữ số? True


## 3. Kiến thức nền

### 3.1. Ghép chữ số thành số: `100*a + 10*b + c`

Ba chữ số `a, b, c` (đầu, giữa, cuối) ghép thành số `100*a + 10*b + c`. Ví dụ `(3, 1, 2)` -> `312`.

In [2]:
a, b, c = 3, 1, 2
print(100 * a + 10 * b + c)       # 312
print(int("".join(map(str, (a, b, c)))))   # cách khác: ghép chuỗi rồi int

312
312


### 3.2. Số chẵn: chữ số cuối chẵn

Số chẵn là số chia hết cho 2: `x % 2 == 0`. Tương đương: **chữ số hàng đơn vị** là `0, 2, 4, 6, 8`. Số đầu không được `0`: nếu `a = 0` thì ta chỉ có số 2 chữ số.

In [3]:
for x in (102, 103, 320, 31):
    print(x, "chẵn?", x % 2 == 0, "| chữ số cuối:", x % 10)

102 chẵn? True | chữ số cuối: 2
103 chẵn? False | chữ số cuối: 3
320 chẵn? True | chữ số cuối: 0
31 chẵn? False | chữ số cuối: 1


### 3.3. `set` để loại trùng và `sorted` để sắp xếp

`set` giữ phần tử duy nhất. `sorted(s)` trả **list** tăng dần.

In [4]:
s = set()
for x in (5, 3, 5, 1, 3):
    s.add(x)
print(s, sorted(s))

{1, 3, 5} [1, 3, 5]


### 3.4. `itertools.permutations(iterable, r)` — hoán vị chọn `r`

`permutations(seq, 3)` sinh mọi bộ 3 phần tử **ở vị trí khác nhau** (không lấy một vị trí hai lần), có tính thứ tự. Nếu `seq` có phần tử trùng giá trị, ta vẫn có các bộ trùng giá trị -> cần `set`.

In [5]:
from itertools import permutations

print(list(permutations([1, 2, 3], 2)))
print(len(list(permutations([2, 2, 8], 3))), "bộ (có trùng giá trị)")
print(set(permutations([2, 2, 8], 3)))

[(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
6 bộ (có trùng giá trị)
{(8, 2, 2), (2, 8, 2), (2, 2, 8)}


### 3.5. `Counter` — đếm mỗi chữ số có bao nhiêu cái

`Counter(digits)[d]` cho số lần chữ số `d` xuất hiện (0 nếu không có). Đây là chìa khóa của cách tối ưu: kiểm tra **một số có "dựng được" từ `digits` không** chỉ bằng việc so sánh số lần.

In [6]:
from collections import Counter

cnt = Counter([2, 2, 8, 8, 2])
print(cnt, cnt[2], cnt[8], cnt[5])
need = Counter([8, 8, 8])
print("cần 888:", all(cnt[d] >= k for d, k in need.items()))

Counter({2: 3, 8: 2}) 3 2 0
cần 888: False


## 4. Ý tưởng nền tảng — tính bằng tay

Ví dụ 1: `digits = [2,1,3,0]`. Chữ số cuối chẵn có thể là `2` hoặc `0`.

- Cuối là `2`: hai chữ số còn lại từ `{1,3,0}`, đầu ≠ 0: `(1,3)->132`, `(3,1)->312`, `(1,0)->102`, `(3,0)->302` (còn `(0,*)` bị loại vì đầu 0) -> `102,132,302,312`.
- Cuối là `0`: hai chữ số còn lại từ `{2,1,3}`: `210, 230, 120, 320, 130, 310` -> sáu số.

Tổng 10 số — khớp đáp án. Ý chính: **chọn chữ số cuối chẵn, chọn đầu khác 0, chọn giữa**, rồi nhớ không dùng chữ số nào quá số lần có.

## 5. Cách 1 — Brute Force: ba vòng lặp theo chỉ số

Chọn ba chỉ số `i, j, k` đôi một khác nhau; ghép `digits[i]digits[j]digits[k]`. Nếu `digits[i] != 0` và `digits[k]` chẵn thì thêm số vào `set`.

In [7]:
def find_even_brute(digits):
    n = len(digits)
    found = set()
    for i in range(n):
        if digits[i] == 0:
            continue                       # chữ số đầu không được 0
        for j in range(n):
            if j == i:
                continue
            for k in range(n):
                if k == i or k == j:
                    continue
                if digits[k] % 2 == 0:     # chữ số cuối chẵn
                    found.add(100 * digits[i] + 10 * digits[j] + digits[k])
    return sorted(found)

print(find_even_brute([2, 1, 3, 0]))
print(find_even_brute([2, 2, 8, 8, 2]))
print(find_even_brute([3, 7, 5]))

[102, 120, 130, 132, 210, 230, 302, 310, 312, 320]
[222, 228, 282, 288, 822, 828, 882]
[]


**Độ phức tạp:** `O(n³)` với `n = len(digits) ≤ 100` -> tối đa ~10⁶ bước: vẫn qua, nhưng là nút thắt vì ta lặp lại cả khi các chữ số **giống hệt nhau** (digits có 100 phần tử nhưng chỉ có 10 giá trị khác nhau!).

Câu hỏi dẫn dắt: *đáp án có tối đa bao nhiêu số?* Các số chẵn trong `[100, 999]` chỉ có 450. Tại sao không **liệt kê 450 ứng viên** và kiểm tra từng ứng viên?

## 6. Ý tưởng bước đột phá — đảo vai trò: liệt kê đáp án, kiểm tra bằng đếm

Thay vì "từ đầu vào sinh số", ta làm ngược lại: **với mỗi số chẵn `x` trong `[100, 998]`**, kiểm tra xem `digits` có đủ chữ số để dựng `x` không.

Quy tắc kiểm tra (invariant): lấy ba chữ số của `x` -> `need = Counter(ba chữ số)`. **Nếu** với mọi chữ số `d` ta có `cnt[d] >= need[d]` **thì** `x` dựng được.

Ưu điểm: tối đa `450 × 3` thao tác, **không phụ thuộc `n`** ngoài bước đếm `O(n)`. Kết quả duyệt tăng dần nên khỏi cần `sort`.

## 7. Các cách tiếp theo

### Cách 2 — `itertools.permutations` + `set`

Tạo mọi hoán vị 3 phần tử, lọc điều kiện, đưa vào `set`. Code ngắn nhất nhưng vẫn `O(n³)` (số hoán vị ~ n·(n-1)·(n-2)).

In [8]:
from itertools import permutations

def find_even_perm(digits):
    found = set()
    for a, b, c in permutations(digits, 3):
        if a != 0 and c % 2 == 0:
            found.add(100 * a + 10 * b + c)
    return sorted(found)

print(find_even_perm([2, 1, 3, 0]))

[102, 120, 130, 132, 210, 230, 302, 310, 312, 320]


In [9]:
from typing import List
from itertools import permutations

class SolutionPermutations:
    def findEvenNumbers(self, digits: List[int]) -> List[int]:
        found = set()
        for a, b, c in permutations(digits, 3):
            if a != 0 and c % 2 == 0:
                found.add(100 * a + 10 * b + c)
        return sorted(found)

print(SolutionPermutations().findEvenNumbers([3, 7, 5]))

[]


Độ phức tạp Cách 2: `O(n³ log n)` (sort tối đa 450 phần tử là nhỏ) thời gian, `O(450)` bộ nhớ cho tập kết quả.

### Cách 3 — Liệt kê 450 số chẵn + so sánh `Counter` (cách tối ưu)

Với mỗi `x` trong `range(100, 1000, 2)`, tách `a, b, c = x // 100, x // 10 % 10, x % 10`, đếm số lần mỗi chữ số trong `x`, rồi so với đếm của `digits`.

In [10]:
from collections import Counter

def find_even_count(digits):
    cnt = Counter(digits)
    ans = []
    for x in range(100, 1000, 2):          # 100, 102, ..., 998 (chỉ số chẵn)
        need = Counter((x // 100, x // 10 % 10, x % 10))
        if all(cnt[d] >= k for d, k in need.items()):
            ans.append(x)
    return ans

print(find_even_count([2, 1, 3, 0]))
print(find_even_count([2, 2, 8, 8, 2]))

[102, 120, 130, 132, 210, 230, 302, 310, 312, 320]
[222, 228, 282, 288, 822, 828, 882]


In [11]:
from typing import List
from collections import Counter

class SolutionCount:
    def findEvenNumbers(self, digits: List[int]) -> List[int]:
        cnt = Counter(digits)
        ans = []
        for x in range(100, 1000, 2):
            need = Counter((x // 100, x // 10 % 10, x % 10))
            if all(cnt[d] >= k for d, k in need.items()):
                ans.append(x)
        return ans

print(SolutionCount().findEvenNumbers([2, 2, 8, 8, 2]))

[222, 228, 282, 288, 822, 828, 882]


Độ phức tạp Cách 3: `O(n + 450·3)`: đếm `digits` `O(n)`, rồi hằng số. Bộ nhớ `O(10)`.

### Cách 4 — Backtracking với mảng đếm 10 ô

Dựng số từng chữ số: chọn hàng trăm (1–9), hàng chục (0–9), hàng đơn vị (0,2,4,6,8); mỗi lần chọn một chữ số `d` chỉ khi `cnt[d] > 0`, rồi trừ `cnt[d]` và quay lui. Duyệt theo thứ tự tăng nên kết quả tự có thứ tự.

In [12]:
def find_even_backtrack(digits):
    cnt = [0] * 10
    for d in digits:
        cnt[d] += 1
    ans = []

    def dfs(pos, cur):
        if pos == 3:
            ans.append(cur)
            return
        for d in range(10):
            if cnt[d] == 0:
                continue
            if pos == 0 and d == 0:
                continue                   # hàng trăm khác 0
            if pos == 2 and d % 2 == 1:
                continue                   # hàng đơn vị chẵn
            cnt[d] -= 1
            dfs(pos + 1, cur * 10 + d)
            cnt[d] += 1                    # hoàn tác (backtrack)

    dfs(0, 0)
    return ans

print(find_even_backtrack([2, 1, 3, 0]))

[102, 120, 130, 132, 210, 230, 302, 310, 312, 320]


In [13]:
from typing import List

class SolutionBacktrack:
    def findEvenNumbers(self, digits: List[int]) -> List[int]:
        cnt = [0] * 10
        for d in digits:
            cnt[d] += 1
        ans = []

        def dfs(pos, cur):
            if pos == 3:
                ans.append(cur)
                return
            for d in range(10):
                if cnt[d] == 0 or (pos == 0 and d == 0) or (pos == 2 and d % 2):
                    continue
                cnt[d] -= 1
                dfs(pos + 1, cur * 10 + d)
                cnt[d] += 1

        dfs(0, 0)
        return ans

print(SolutionBacktrack().findEvenNumbers([3, 7, 5]))

[]


Độ phức tạp Cách 4: tối đa `9·10·5 = 450` lá, mỗi bước `O(10)` -> hằng số; `O(n)` để đếm đầu vào. Thứ tự tăng dần có sẵn.

## 8. TRACE từng bước

Trace Cách 3 trên `digits = [2,2,8,8,2]`: `cnt = {2:3, 8:2}`.

| x | chữ số cần | đủ? |
|---|---|---|
| 222 | 2×3 | có (cnt[2]=3) |
| 228 | 2×2, 8×1 | có |
| 288 | 2×1, 8×2 | có |
| 888 | 8×3 | **không** (cnt[8]=2) |
| 102 | 1,0,2 | không (không có 1) |

In [14]:
def trace_count(digits, candidates):
    cnt = Counter(digits)
    print("cnt =", dict(cnt))
    for x in candidates:
        need = Counter((x // 100, x // 10 % 10, x % 10))
        detail = {d: (k, cnt[d]) for d, k in need.items()}   # chữ số: (cần, có)
        ok = all(cnt[d] >= k for d, k in need.items())
        print(f"x={x}: cần/có = {detail} -> {'DỰNG ĐƯỢC' if ok else 'thiếu'}")

trace_count([2, 2, 8, 8, 2], [222, 228, 288, 888, 102])

cnt = {2: 3, 8: 2}
x=222: cần/có = {2: (3, 3)} -> DỰNG ĐƯỢC
x=228: cần/có = {2: (2, 3), 8: (1, 2)} -> DỰNG ĐƯỢC
x=288: cần/có = {2: (1, 3), 8: (2, 2)} -> DỰNG ĐƯỢC
x=888: cần/có = {8: (3, 2)} -> thiếu
x=102: cần/có = {1: (1, 0), 0: (1, 0), 2: (1, 3)} -> thiếu


In [15]:
def trace_backtrack(digits):
    cnt = [0] * 10
    for d in digits:
        cnt[d] += 1
    def dfs(pos, cur, depth):
        if pos == 3:
            print("  " * depth + f"=> số {cur}")
            return
        for d in range(10):
            if cnt[d] == 0 or (pos == 0 and d == 0) or (pos == 2 and d % 2):
                continue
            print("  " * depth + f"pos={pos} chọn {d}, còn lại={cnt}")
            cnt[d] -= 1
            dfs(pos + 1, cur * 10 + d, depth + 1)
            cnt[d] += 1
    dfs(0, 0, 0)

trace_backtrack([1, 2, 2])

pos=0 chọn 1, còn lại=[0, 1, 2, 0, 0, 0, 0, 0, 0, 0]
  pos=1 chọn 2, còn lại=[0, 0, 2, 0, 0, 0, 0, 0, 0, 0]
    pos=2 chọn 2, còn lại=[0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
      => số 122
pos=0 chọn 2, còn lại=[0, 1, 2, 0, 0, 0, 0, 0, 0, 0]
  pos=1 chọn 1, còn lại=[0, 1, 1, 0, 0, 0, 0, 0, 0, 0]
    pos=2 chọn 2, còn lại=[0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
      => số 212
  pos=1 chọn 2, còn lại=[0, 1, 1, 0, 0, 0, 0, 0, 0, 0]


## 9. Code nộp LeetCode

Dùng Cách 3 (liệt kê 450 số chẵn + so đếm): ngắn, nhanh, kết quả đã tăng dần.

In [16]:
from typing import List, Optional
from collections import Counter

class Solution:
    def findEvenNumbers(self, digits: List[int]) -> List[int]:
        cnt = Counter(digits)
        ans = []
        for x in range(100, 1000, 2):
            need = Counter((x // 100, x // 10 % 10, x % 10))
            if all(cnt[d] >= k for d, k in need.items()):
                ans.append(x)
        return ans

print(Solution().findEvenNumbers([2, 1, 3, 0]))

[102, 120, 130, 132, 210, 230, 302, 310, 312, 320]


## 10. Vì sao thuật toán đúng?

- **Đủ (completeness):** mọi đáp án hợp lệ đều là số chẵn trong `[100, 998]`, nên nằm trong danh sách ứng viên `range(100, 1000, 2)`.
- **Đúng (soundness):** `x` dựng được từ `digits` **khi và chỉ khi** với mỗi chữ số `d`, số lần `d` xuất hiện trong `x` không vượt quá số lần có trong `digits` (ta có thể chọn đúng các vị trí tương ứng, vì thứ tự ghép là tùy ý).
- **Không trùng:** mỗi `x` được xét đúng một lần.
- **Tăng dần:** `range` tăng dần nên `ans` tự có thứ tự.

Đây là phép biến đổi "đảo chiều": thay vì sinh từ đầu vào (rất nhiều hoán vị trùng), ta **duyệt không gian đáp án nhỏ cố định** rồi xác minh.

In [17]:
# Kiểm tra: mọi số trả về đều chẵn, đủ 3 chữ số, tăng dần, và dựng được từ digits
import random
random.seed(7)
for _ in range(300):
    ds = [random.randint(0, 9) for _ in range(random.randint(3, 8))]
    res = Solution().findEvenNumbers(ds)
    assert res == sorted(set(res))
    for x in res:
        assert 100 <= x <= 999 and x % 2 == 0
        need = Counter(map(int, str(x))); have = Counter(ds)
        assert all(have[d] >= k for d, k in need.items())
print("invariant OK")

invariant OK


## 11. Phân tích Time / Space Complexity

`n = len(digits)`; số ứng viên cố định `450`.

| Cách | Time | Space |
|---|---|---|
| 1. Ba vòng lặp chỉ số | `O(n³)` | `O(450)` |
| 2. `permutations` + set | `O(n³ log n)` | `O(450)` |
| 3. Liệt kê 450 + Counter | `O(n + 450)` ≈ `O(n)` | `O(1)` |
| 4. Backtracking mảng đếm | `O(n + 450·10)` ≈ `O(n)` | `O(1)` (độ sâu 3) |

Với `n ≤ 100`, cả bốn đều chạy nhanh trên LeetCode; Cách 3 và 4 mở rộng tốt nhất nếu `n` rất lớn.

## 12. Bộ test

Ví dụ đề, edge case (chỉ có số lẻ, chỉ có 0, đúng 3 chữ số) và đối chiếu ngẫu nhiên cả 4 cách.

In [18]:
ALL = [find_even_brute, find_even_perm, find_even_count, find_even_backtrack]
cases = [
    ([2, 1, 3, 0], [102, 120, 130, 132, 210, 230, 302, 310, 312, 320]),
    ([2, 2, 8, 8, 2], [222, 228, 282, 288, 822, 828, 882]),
    ([3, 7, 5], []),
    ([0, 0, 0], []),            # 000 có số 0 đầu
    ([1, 2, 0], [102, 120, 210]),
    ([4, 4, 4], [444]),
    ([1, 1, 1], []),            # không có chữ số chẵn
    ([0, 2, 0], [200]),
]
for ds, exp in cases:
    for f in ALL:
        assert f(ds) == exp, (f.__name__, ds, f(ds), exp)
    assert Solution().findEvenNumbers(ds) == exp
print("Tất cả test ví dụ + edge case qua:", len(cases), "ca x", len(ALL) + 1, "cách")

Tất cả test ví dụ + edge case qua: 8 ca x 5 cách


In [19]:
import random
random.seed(2094)
for _ in range(1500):
    n = random.randint(3, 9)
    ds = [random.randint(0, 9) for _ in range(n)]
    base = find_even_brute(ds)
    assert find_even_perm(ds) == base
    assert find_even_count(ds) == base
    assert find_even_backtrack(ds) == base
    assert Solution().findEvenNumbers(ds) == base
print("1500 test ngẫu nhiên: các cách khớp nhau")

1500 test ngẫu nhiên: các cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên loại số trùng (trả về list có phần tử lặp)

Khi `digits` có chữ số trùng, nhiều hoán vị cho cùng một số.

In [20]:
def wrong_duplicates(digits):
    ans = []
    for a, b, c in permutations(digits, 3):
        if a != 0 and c % 2 == 0:
            ans.append(100 * a + 10 * b + c)
    return sorted(ans)

print("sai:", wrong_duplicates([2, 2, 8, 8, 2]))
print("đúng:", Solution().findEvenNumbers([2, 2, 8, 8, 2]))

sai: [222, 222, 222, 222, 222, 222, 228, 228, 228, 228, 228, 228, 228, 228, 228, 228, 228, 228, 282, 282, 282, 282, 282, 282, 282, 282, 282, 282, 282, 282, 288, 288, 288, 288, 288, 288, 822, 822, 822, 822, 822, 822, 822, 822, 822, 822, 822, 822, 828, 828, 828, 828, 828, 828, 882, 882, 882, 882, 882, 882]
đúng: [222, 228, 282, 288, 822, 828, 882]


Sửa: dùng `set` trước khi `sorted`.

### Lỗi 2: Cho phép chữ số 0 đứng đầu

In [21]:
def wrong_leading_zero(digits):
    found = set()
    for a, b, c in permutations(digits, 3):
        if c % 2 == 0:                       # quên a != 0
            found.add(100 * a + 10 * b + c)
    return sorted(found)

print("sai:", wrong_leading_zero([1, 2, 0]))
print("đúng:", Solution().findEvenNumbers([1, 2, 0]))

sai: [12, 102, 120, 210]
đúng: [102, 120, 210]


`(0, 1, 2)` cho `12` — chỉ là số 2 chữ số. Phải có `a != 0`.

### Lỗi 3: Dùng lại cùng một chữ số quá số lần có (dùng `product` thay vì hoán vị theo vị trí)

In [22]:
from itertools import product

def wrong_reuse(digits):
    found = set()
    for a, b, c in product(set(digits), repeat=3):    # mỗi chữ số dùng thoải mái
        if a != 0 and c % 2 == 0:
            found.add(100 * a + 10 * b + c)
    return sorted(found)

print("sai :", wrong_reuse([3, 2, 8]))
print("đúng:", Solution().findEvenNumbers([3, 2, 8]))

sai : [222, 228, 232, 238, 282, 288, 322, 328, 332, 338, 382, 388, 822, 828, 832, 838, 882, 888]
đúng: [238, 328, 382, 832]


`product` cho phép `222`, `388`... dù chỉ có **một** chữ số 2 hoặc 8. Phải kiểm soát số lần dùng (vị trí khác nhau hoặc `Counter`).

### Lỗi 4 (thêm): Quên chữ số cuối phải chẵn khi liệt kê bước nhảy

`range(100, 1000)` (bước 1) rồi quên điều kiện `x % 2 == 0` sẽ trả cả số lẻ. Dùng `range(100, 1000, 2)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "không gian đáp án **nhỏ và cố định**" (ở đây 450 số) -> hãy **duyệt đáp án** và kiểm tra tính hợp lệ.
- Thấy "dùng mỗi phần tử tối đa số lần có" -> **Counter** và so sánh `need <= have`.
- Thấy "kết quả không trùng, tăng dần" -> `set` + `sorted`, hoặc duyệt theo thứ tự tăng.
- Số chữ số cố định (3) -> các bước `// 100`, `// 10 % 10`, `% 10` để tách chữ số.
- Với hoán vị ngắn (3) thì `itertools.permutations` rất tiện khi cần viết nhanh.

## 15. Biến thể của bài

1. Tạo số có **`k` chữ số** (tổng quát hóa) — khi `k` lớn, backtracking vẫn dùng được.
2. Đếm **số lượng** số hợp lệ thay vì liệt kê.
3. Yêu cầu số **chia hết cho 5** hoặc cho 3 (tổng chữ số chia hết cho 3).
4. Cho phép chữ số 0 đứng đầu (số có thể là `012`) — bỏ điều kiện `a != 0`.
5. LeetCode 1295, 1346... luyện thêm tư duy chữ số; LeetCode 46/47 (Permutations I/II) để học hoán vị có trùng.

In [23]:
def count_even_numbers(digits):
    return len(find_even_count(digits))

def find_divisible_by_5(digits):
    cnt = Counter(digits)
    return [x for x in range(100, 1000, 5)
            if all(cnt[d] >= k for d, k in Counter((x // 100, x // 10 % 10, x % 10)).items())]

print(count_even_numbers([2, 1, 3, 0]))
print(find_divisible_by_5([5, 0, 1]))

10
[105, 150, 510]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng chỉ số | Chọn 3 vị trí khác nhau | `O(n³)` | `O(450)` | Học cơ bản, `n` nhỏ |
| 2. `permutations` | Sinh hoán vị bằng thư viện | `O(n³ log n)` | `O(450)` | Viết nhanh khi `n` nhỏ |
| 3. Liệt kê 450 + Counter | Duyệt đáp án, kiểm tra bằng đếm | `O(n)` | `O(1)` | **Nên dùng**, mở rộng tốt |
| 4. Backtracking | Dựng từng chữ số, trừ/hoàn tác đếm | `O(n)` | `O(1)` | Khi `k` chữ số tổng quát |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 — Đoán output:** `Solution().findEvenNumbers([2, 4, 6])` trả về gì?

**Bài 2 — Tính tay:** `digits = [1, 0, 4]`. Liệt kê mọi số hợp lệ.

**Bài 3 — Điền TODO:** hoàn thành `can_build(x, digits)` trả `True` nếu dựng được `x` từ `digits`.

**Bài 4 — Tự viết lại:** viết `exercise_find_even(digits)` theo cách riêng (ví dụ `permutations` + `set`).

**Bài 5 — Mini challenge:** viết `count_k_digit_even(digits, k)` đếm số chẵn `k` chữ số (không 0 đầu) dựng được, `1 <= k <= len(digits)`.

Các hàm chỉ được định nghĩa, không gọi, để chạy toàn bộ notebook không gây lỗi.

In [24]:
def can_build(x, digits):
    # TODO: so sánh Counter(str(x)) với Counter(digits)
    pass

# assert can_build(102, [2, 1, 0, 9]) is True

In [25]:
def exercise_find_even(digits):
    # TODO: viết theo cách riêng; trả về list tăng dần không trùng
    pass

# assert exercise_find_even([1, 0, 4]) == [104, 140, 410]

In [26]:
def count_k_digit_even(digits, k):
    # TODO: backtracking với mảng đếm 10 ô; hàng đầu != 0, hàng cuối chẵn
    pass

# assert count_k_digit_even([1, 2, 0], 3) == 3

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Chữ số cuối phải chẵn | Cả ba đều chẵn nên mọi hoán vị đều ok | `[246, 264, 426, 462, 624, 642]` |
| 2 | Cuối chẵn: 0 hoặc 4 | Đầu không phải 0 | `104, 140, 410` |
| 3 | `Counter(map(int, str(x)))` | So với `Counter(digits)` từng chữ số | `all(have[d] >= k for ...)` |
| 4 | `permutations(digits, 3)` | Lọc `a != 0 and c % 2 == 0` | `sorted(set(...))` |
| 5 | Khái quát `pos == 3` thành `pos == k` | Chỉ `pos == k-1` cần chẵn | Dùng `cnt` 10 ô như Cách 4 |

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[246, 264, 426, 462, 624, 642]`. **Bài 2:** `[104, 140, 410]`.

In [27]:
assert Solution().findEvenNumbers([2, 4, 6]) == [246, 264, 426, 462, 624, 642]
assert Solution().findEvenNumbers([1, 0, 4]) == [104, 140, 410]

def can_build_answer(x, digits):
    need, have = Counter(map(int, str(x))), Counter(digits)
    return all(have[d] >= k for d, k in need.items())

def exercise_find_even_answer(digits):
    return sorted({100 * a + 10 * b + c for a, b, c in permutations(digits, 3) if a != 0 and c % 2 == 0})

def count_k_digit_even_answer(digits, k):
    cnt = [0] * 10
    for d in digits:
        cnt[d] += 1
    def dfs(pos):
        if pos == k:
            return 1
        total = 0
        for d in range(10):
            if cnt[d] == 0 or (pos == 0 and d == 0) or (pos == k - 1 and d % 2):
                continue
            cnt[d] -= 1
            total += dfs(pos + 1)
            cnt[d] += 1
        return total
    return dfs(0)

assert can_build_answer(102, [2, 1, 0, 9]) is True
assert can_build_answer(888, [8, 8, 1]) is False
assert exercise_find_even_answer([1, 0, 4]) == [104, 140, 410]
assert count_k_digit_even_answer([1, 2, 0], 3) == 3
assert count_k_digit_even_answer([2, 1, 3, 0], 3) == 10
print("Đáp án bài tập: tất cả assert đều qua")

Đáp án bài tập: tất cả assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template**

```python
from collections import Counter
class Solution:
    def findEvenNumbers(self, digits):
        cnt = Counter(digits)
        ans = []
        for x in range(100, 1000, 2):
            need = Counter((x // 100, x // 10 % 10, x % 10))
            if all(cnt[d] >= k for d, k in need.items()):
                ans.append(x)
        return ans
```

**Checklist**

- [ ] Số có đúng 3 chữ số, **không** bắt đầu bằng 0, **chẵn**.
- [ ] Mỗi chữ số không dùng nhiều hơn số lần có trong `digits`.
- [ ] Kết quả **không trùng** và **tăng dần**.
- [ ] Đã test `[2,1,3,0]`, `[2,2,8,8,2]`, `[3,7,5]` và các biên (`[0,0,0]`, `[1,1,1]`).
- [ ] Biết đánh đổi: sinh từ đầu vào `O(n³)` hay duyệt 450 ứng viên `O(n)`.